In [ ]:
# ==========================================
# 1. INSTALLATIONS & IMPORTS
# ==========================================
# Install necessary libraries (Quiet mode)
!pip install -q transformers datasets torch scikit-learn accelerate

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    RobertaPreTrainedModel,  # <--- CHANGED
    RobertaModel,            # <--- CHANGED
    Trainer,
    TrainingArguments,
    EvalPrediction
)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. CONFIGURATION
# ==========================================
MODEL_ID = "roberta-base" # <--- CHANGED
MAX_LENGTH = 256        # Length of (Question + 1 Chunk)
CHUNK_STRIDE = 160       # Overlap
MAX_CHUNKS = 16       # Max chunks per answer
BATCH_SIZE = 8
FOCAL_GAMMA = 2.0       # Focusing parameter

# ==========================================
# 3. DATA LOADING & WEIGHT CALCULATION
# ==========================================
print("\n--- Loading Data ---")
dataset = load_dataset("ailsntua/QEvasion")

# Map Labels
labels_list = dataset['train'].unique('clarity_label')
num_labels = len(labels_list)
label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for i, l in enumerate(labels_list)}

print(f"Labels: {label2id}")

def encode_labels(batch):
    return {"labels": label2id[batch['clarity_label']]}

dataset = dataset.map(encode_labels)

# Calculate Class Weights
print("\n--- Calculating Class Weights ---")
train_labels = dataset['train']['labels']
label_counts = Counter(train_labels)
total_samples = len(train_labels)

class_weights = []
for i in range(num_labels):
    count = label_counts.get(i, 1)
    weight = total_samples / (num_labels * count)
    class_weights.append(weight)

# Convert to Tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class Weights: {class_weights_tensor}")

# ==========================================
# 4. CUSTOM DATASET (THE CHUNKER)
# ==========================================
class MILChunkingDataset(Dataset):
    """
    Takes a row (Question, Long Answer) and splits it into multiple
    overlapping chunks ON THE FLY.
    ADAPTED FOR ROBERTA: No token_type_ids.
    """
    def __init__(self, hf_dataset, tokenizer, max_len, stride, max_chunks):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride
        self.max_chunks = max_chunks

        # Silence length warnings
        self.tokenizer.model_max_length = 1_000_000

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        question = row['question']
        long_answer = row['interview_answer']
        label = row['labels']

        # 1. Tokenize separately to get raw IDs
        q_ids = self.tokenizer.encode(question, add_special_tokens=False)
        a_ids = self.tokenizer.encode(long_answer, add_special_tokens=False)

        # Calculate space available for answer tokens
        # RoBERTa: <s> Q </s> A </s> (3 special tokens)
        tokens_for_answer = self.max_len - (len(q_ids) + 3)

        if tokens_for_answer < 10:
            tokens_for_answer = 50
            q_ids = q_ids[:self.max_len - 60]

        # 2. Create Sliding Windows
        chunk_inputs = []
        chunk_masks = []

        if len(a_ids) == 0:
            windows = [[]]
        else:
            windows = [
                a_ids[i : i + tokens_for_answer]
                for i in range(0, len(a_ids), self.stride)
            ]
            windows = windows[:self.max_chunks]

        # 3. Build Inputs Manually
        cls_id = self.tokenizer.cls_token_id # <s>
        sep_id = self.tokenizer.sep_token_id # </s>
        pad_id = self.tokenizer.pad_token_id # <pad>

        for window in windows:
            # Structure: <s> Q </s> Window </s>
            input_ids = [cls_id] + q_ids + [sep_id] + window + [sep_id]

            # Create Mask (1 for real tokens)
            attention_mask = [1] * len(input_ids)

            # Padding
            padding_length = self.max_len - len(input_ids)
            if padding_length > 0:
                input_ids = input_ids + [pad_id] * padding_length
                attention_mask = attention_mask + [0] * padding_length
            else:
                input_ids = input_ids[:self.max_len]
                attention_mask = attention_mask[:self.max_len]

            chunk_inputs.append(torch.tensor(input_ids, dtype=torch.long))
            chunk_masks.append(torch.tensor(attention_mask, dtype=torch.long))
            # NOTE: No token_type_ids for RoBERTa

        # 4. Stack
        return {
            "input_ids": torch.stack(chunk_inputs),       # [N_Chunks, Seq_Len]
            "attention_mask": torch.stack(chunk_masks),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# ==========================================
# 5. DATA COLLATOR (THE PADDER)
# ==========================================
@dataclass
class MILDataCollator:
    """
    Pads the 'Bag' dimension so all items in a batch have the same number of chunks.
    """
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        max_chunks_batch = max(f["input_ids"].shape[0] for f in features)

        batch_input_ids = []
        batch_masks = []
        batch_labels = []

        for f in features:
            source_ids = f["input_ids"]
            source_mask = f["attention_mask"]
            num_chunks, seq_len = source_ids.shape

            pad_chunks = max_chunks_batch - num_chunks
            if pad_chunks > 0:
                pad_ids = torch.zeros((pad_chunks, seq_len), dtype=torch.long)
                pad_mask = torch.zeros((pad_chunks, seq_len), dtype=torch.long)

                # RoBERTa uses pad_token_id=1, but for entire chunks we can just use 0s
                # as long as the mask is 0.

                padded_ids = torch.cat([source_ids, pad_ids], dim=0)
                padded_mask = torch.cat([source_mask, pad_mask], dim=0)
            else:
                padded_ids = source_ids
                padded_mask = source_mask

            batch_input_ids.append(padded_ids)
            batch_masks.append(padded_mask)
            batch_labels.append(f["labels"])

        return {
            "input_ids": torch.stack(batch_input_ids), # [Batch, Max_Chunks, Seq_Len]
            "attention_mask": torch.stack(batch_masks),
            "labels": torch.stack(batch_labels)
        }

# ==========================================
# 6. CUSTOM MODEL (ROBERTA + ATTENTION MIL)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        pt = torch.exp(-ce_loss)
        pt = torch.clamp(pt, min=1e-8, max=1.0 - 1e-8)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class RobertaForMIL(RobertaPreTrainedModel): # <--- Inherits from RoBERTa
    def __init__(self, config, class_weights=None, focal_gamma=2.0):
        super().__init__(config)
        self.num_labels = config.num_labels

        # RoBERTa Body
        self.roberta = RobertaModel(config) # <--- RoBERTa Model

        # Attention Mechanism
        self.attention_layer = nn.Linear(config.hidden_size, 1)

        # Classifier Head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Loss Setup
        self.class_weights = class_weights
        self.focal_gamma = focal_gamma
        self.post_init()

    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        **kwargs
    ) -> SequenceClassifierOutput:

        # input_ids: [Batch, Chunks, Seq_Len]
        batch_size, num_chunks, seq_len = input_ids.shape

        # 1. Flatten: [Batch * Chunks, Seq_Len]
        flat_input_ids = input_ids.view(-1, seq_len)
        flat_mask = attention_mask.view(-1, seq_len)

        # 2. Pass through RoBERTa (No token_type_ids)
        outputs = self.roberta(
            input_ids=flat_input_ids,
            attention_mask=flat_mask
        )

        # 3. Extract [CLS] Token (First token)
        cls_output = outputs.last_hidden_state[:, 0, :]

        # --- ATTENTION POOLING START ---

        # Calculate raw attention scores [Batch * Chunks, 1]
        attn_scores = self.attention_layer(cls_output)

        # Reshape to [Batch, Chunks]
        attn_scores = attn_scores.view(batch_size, num_chunks)

        # Mask padding chunks (where attention_mask is 0 for the whole chunk)
        chunk_mask = torch.any(attention_mask > 0, dim=-1)

        # Set attention scores of padding chunks to -infinity
        min_val = -65000.0
        attn_scores = attn_scores.masked_fill(~chunk_mask, min_val)

        # Softmax
        attn_weights = F.softmax(attn_scores, dim=1)

        # Weighted Average
        cls_output_reshaped = cls_output.view(batch_size, num_chunks, -1)
        context_vector = torch.sum(cls_output_reshaped * attn_weights.unsqueeze(-1), dim=1)

        # --- ATTENTION POOLING END ---

        # 4. Classifier Head
        context_vector = self.dropout(context_vector)
        logits = self.classifier(context_vector)

        loss = None
        if labels is not None:
            if self.class_weights is not None:
                self.class_weights = self.class_weights.to(logits.device)

            loss_fct = FocalLoss(gamma=self.focal_gamma, weight=self.class_weights)
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

# ==========================================
# 7. TRAINING EXECUTION
# ==========================================

# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Create Datasets
print("\n--- Creating Datasets ---")
train_ds = MILChunkingDataset(dataset['train'], tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS)
test_ds = MILChunkingDataset(dataset['test'], tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS)

# Initialize Model
print("\n--- Initializing RoBERTa Model ---")
model = RobertaForMIL.from_pretrained(
    MODEL_ID,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    class_weights=class_weights_tensor,
    focal_gamma=FOCAL_GAMMA
)

# Metrics
def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    f1 = f1_score(p.label_ids, preds, average='macro')
    return {"accuracy": acc, "f1_macro": f1}

# Training Arguments
training_args = TrainingArguments(
    output_dir="./Roberta_MIL_Attention",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=MILDataCollator(),
    compute_metrics=compute_metrics
)

# Start Training
print("\n--- Starting Training ---")
trainer.train()

# ==========================================
# 8. FINAL EVALUATION
# ==========================================
print("\n--- Final Evaluation on Test Set ---")
results = trainer.predict(test_ds)
y_preds = np.argmax(results.predictions, axis=1)
y_true = results.label_ids

print("\nClassification Report:")
print(classification_report(y_true, y_preds, target_names=labels_list))

# Save
trainer.save_model("./Final_Roberta_MIL_Attn_Model")
tokenizer.save_pretrained("./Final_Roberta_MIL_Attn_Model")
print("\nModel saved to ./Final_Roberta_MIL_Attn_Model")